# Mr.X R-GNN - Stage X1 warm-start train

Notebook per Kaggle Notebook, pronto per **Save & Commit**.

Input atteso: dataset Kaggle chiamato `MrXlogged` che contiene `results.zip` prodotto dal logger. Il codice cerca comunque `results.zip` ricorsivamente in `/kaggle/input`, quindi funziona anche se Kaggle trasforma il nome del dataset in slug lowercase.

Output in `/kaggle/working/mrx_bc_checkpoints`:
- `mrx_rgnn_bc_best_*.pt`
- `mrx_rgnn_bc_last_*.pt`
- `mrx_rgnn_bc_history_*.csv`
- `mrx_rgnn_bc_curves_*.png`

Se il modello supera la validation, promuovilo nel repo rinominandolo in `Notebook/Models/mrx/mrx_bc_vXXX.pt` e aggiungendo la entry in `Notebook/Registry/model_registry.json`. Non sovrascrivere i vecchi checkpoint promossi.

## Feature recap

### Node features Mr.X, 14 dinamiche per nodo

Queste sono le feature salvate in `tensors_part_*.npz` come `node_dyn`, shape `(samples, 199, 14)`:

| Indici | Feature | Significato |
|---:|---|---|
| 0 | `is_mrx` | 1 solo sul vero nodo corrente di Mr.X. Per Mr.X questa informazione e' lecita. |
| 1-5 | `is_detective_k` | occupancy one-hot dei 5 detective. |
| 6 | `belief[v]` | belief dei detective su Mr.X. Mr.X puo' stimare cio' che i detective credono. |
| 7 | `is_revealed_mrx_node` | nodi in cui Mr.X e' gia' stato rivelato. |
| 8 | `min_dist_to_detectives` | distanza minima del nodo da un detective. |
| 9-13 | `dist_to_detective_k` | distanza del nodo da ciascun detective. |

Le feature statiche nodo stanno in `static_graph.npz` come `node_static`, shape `(199, 5)`:

| Indici | Feature | Significato |
|---:|---|---|
| 0-3 | relation degrees | grado taxi, bus, underground, water. |
| 4 | `has_underground` | 1 se il nodo tocca almeno un arco underground. |

Il modello concatena dinamiche e statiche prima dei layer R-GCN.

### Global features Mr.X, 28 scalari

`glob`, shape `(samples, 28)`:

| Indici | Feature | Significato |
|---:|---|---|
| 0 | turn normalizzato | `game.turn / 22`. |
| 1 | turns to next reveal | distanza dal prossimo reveal divisa per 5, oppure negativa dopo l'ultimo reveal. |
| 2 | is current reveal turn | flag sul turno corrente. |
| 3 | is next reveal turn | flag sul prossimo turno. |
| 4-7 | Mr.X tickets | taxi, bus, underground, water normalizzati. |
| 8-22 | detective tickets | 5 detective x 3 ticket, normalizzati. |
| 23-27 | last Mr.X ticket one-hot | `none/taxi/bus/underground/water`. |

### Action space Mr.X

L'azione e' ticket-specifica: `(destination, ticket)`. La maschera legale ha shape `(199, 4)` e viene flattenata in `199 * 4 = 796` azioni:

`flat_action = (destination - 1) * 4 + relation_idx` con relation order `taxi, bus, underground, water`.

Questo e' diverso dai detective: per loro l'action head sceglieva solo il vicino-destinazione, perche' il ticket usato veniva poi risolto da `Game.use_ticket`. Per Mr.X il ticket e' parte del target, perche' cambia l'osservazione pubblica lasciata ai detective.

### Differenza rispetto alle feature dei detective

La GNN detective non vede il vero `mrx_pos`: vede `belief`, ticket osservato e posizioni rivelate. La GNN Mr.X invece vede `is_mrx`, perche' Mr.X conosce sempre la propria posizione reale. Inoltre Mr.X vede anche la belief dei detective come modello dell'avversario, mentre i detective non ricevono la traiettoria nascosta completa di Mr.X per evitare leakage.

## 1. Setup

In [ ]:
%%bash
set -e
python -m pip install -q numpy pandas pyarrow tqdm matplotlib scipy networkx || true
mkdir -p /kaggle/working/mrx_bc_checkpoints /kaggle/working/mrxlogged_extracted
ls /kaggle/input || true


## 2. Locate and load dataset

In [ ]:
import glob
import json
import os
import random
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset

KAGGLE_INPUT = Path('/kaggle/input')
WORK_DIR = Path('/kaggle/working')
EXTRACT_DIR = WORK_DIR / 'mrxlogged_extracted'
OUT_DIR = WORK_DIR / 'mrx_bc_checkpoints'
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

zip_candidates = sorted(KAGGLE_INPUT.glob('**/results.zip'))
zip_candidates += sorted(KAGGLE_INPUT.glob('**/*results*.zip'))
zip_candidates = sorted(set(zip_candidates))

if zip_candidates:
    zip_path = zip_candidates[-1]
    print('Using results zip:', zip_path)
    marker = EXTRACT_DIR / '.extracted'
    if not marker.exists():
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(EXTRACT_DIR)
        marker.write_text(str(zip_path), encoding='utf-8')
else:
    print('No results.zip found; looking for extracted mrx_bc_logs under /kaggle/input')

run_dirs = []
run_dirs += sorted(EXTRACT_DIR.glob('**/mrx_bc_logs/mrx_bc_*'))
run_dirs += sorted(KAGGLE_INPUT.glob('**/mrx_bc_logs/mrx_bc_*'))
run_dirs += sorted(KAGGLE_INPUT.glob('**/mrx_bc_*'))
run_dirs = [p for p in sorted(set(run_dirs)) if p.is_dir()]
assert run_dirs, 'Dataset MrXlogged non trovato: serve results.zip con mrx_bc_logs/mrx_bc_*.'

DATA_DIR = run_dirs[-1]
print('Data dir:', DATA_DIR)

manifest_path = DATA_DIR / 'manifest.json'
assert manifest_path.exists(), f'Missing manifest: {manifest_path}'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print('Logger sanity:')
print(json.dumps(manifest.get('sanity', {}), indent=2))

sample_paths = sorted(DATA_DIR.glob('samples_part_*.parquet'))
tensor_paths = sorted(DATA_DIR.glob('tensors_part_*.npz'))
assert sample_paths and tensor_paths, 'Missing samples_part_*.parquet or tensors_part_*.npz.'
assert len(sample_paths) == len(tensor_paths), (len(sample_paths), len(tensor_paths))
print(f'Parts: {len(sample_paths)}')


In [ ]:
static = np.load(DATA_DIR / 'static_graph.npz', allow_pickle=True)
DENSE_ADJ = static['dense_adj'].astype(np.float32)
NODE_STATIC = static['node_static'].astype(np.float32)
RELATIONS = tuple(str(x) for x in static['relations']) if 'relations' in static else ('taxi', 'bus', 'underground', 'water')
N_NODES = int(DENSE_ADJ.shape[1])
N_REL = int(DENSE_ADJ.shape[0])
ACTION_DIM = N_NODES * N_REL
NODE_DYN_DIM = int(static['node_dyn_dim'][0]) if 'node_dyn_dim' in static else 14
GLOBAL_DIM = int(static['global_dim'][0]) if 'global_dim' in static else 28

sample_frames = []
node_dyn_parts = []
glob_parts = []
legal_parts = []
target_parts = []

for sp, tp in tqdm(list(zip(sample_paths, tensor_paths)), desc='Loading parts'):
    df = pd.read_parquet(sp)
    arr = np.load(tp)
    sample_frames.append(df)
    node_dyn_parts.append(arr['node_dyn'])
    glob_parts.append(arr['glob'].astype(np.float32))
    legal_parts.append(arr['legal_action_mask'].reshape(-1, ACTION_DIM).astype(np.bool_))
    target_parts.append(arr['target_action'].astype(np.int64))

samples_df = pd.concat(sample_frames, ignore_index=True)
node_dyn = np.concatenate(node_dyn_parts, axis=0)
glob_features = np.concatenate(glob_parts, axis=0)
legal_mask = np.concatenate(legal_parts, axis=0)
target_action = np.concatenate(target_parts, axis=0)
returns_raw = samples_df['return_to_go'].astype(np.float32).to_numpy()
mrx_pos_idx = node_dyn[:, :, 0].argmax(axis=1).astype(np.int64)

n = len(target_action)
assert node_dyn.shape == (n, N_NODES, NODE_DYN_DIM), node_dyn.shape
assert glob_features.shape == (n, GLOBAL_DIM), glob_features.shape
assert legal_mask.shape == (n, ACTION_DIM), legal_mask.shape
assert returns_raw.shape == (n,), returns_raw.shape
assert np.all(legal_mask[np.arange(n), target_action]), 'Some targets are outside legal mask.'

print('Samples:', n)
print('node_dyn:', node_dyn.shape, node_dyn.dtype)
print('glob:', glob_features.shape, glob_features.dtype)
print('legal_mask:', legal_mask.shape, legal_mask.dtype)
print('target ticket counts:')
display(samples_df['target_ticket'].value_counts())


## 3. Split and dataset

In [ ]:
SEED = 132159
VAL_FRAC = 0.10
BATCH_SIZE = 256
NUM_WORKERS = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

indices = np.arange(n)
rng = np.random.default_rng(SEED)
rng.shuffle(indices)
val_size = max(1, int(n * VAL_FRAC))
val_idx = indices[:val_size]
train_idx = indices[val_size:]

return_mean = float(returns_raw[train_idx].mean())
return_std = float(returns_raw[train_idx].std() + 1e-6)
returns_norm = ((returns_raw - return_mean) / return_std).astype(np.float32)

print('Train:', len(train_idx), 'Val:', len(val_idx))
print('Return mean/std:', return_mean, return_std)


class MrXBCDataset(Dataset):
    def __init__(self, node_dyn, glob_features, legal_mask, target_action, returns_norm, mrx_pos_idx):
        self.node_dyn = node_dyn
        self.glob_features = glob_features
        self.legal_mask = legal_mask
        self.target_action = target_action
        self.returns_norm = returns_norm
        self.mrx_pos_idx = mrx_pos_idx

    def __len__(self):
        return len(self.target_action)

    def __getitem__(self, idx):
        return {
            'node_dyn': torch.from_numpy(self.node_dyn[idx].astype(np.float32, copy=False)),
            'glob': torch.from_numpy(self.glob_features[idx]),
            'legal_mask': torch.from_numpy(self.legal_mask[idx]),
            'target_action': torch.tensor(self.target_action[idx], dtype=torch.long),
            'return_to_go': torch.tensor(self.returns_norm[idx], dtype=torch.float32),
            'mrx_pos_idx': torch.tensor(self.mrx_pos_idx[idx], dtype=torch.long),
        }


full_ds = MrXBCDataset(node_dyn, glob_features, legal_mask, target_action, returns_norm, mrx_pos_idx)
train_loader = DataLoader(
    Subset(full_ds, train_idx.tolist()),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    Subset(full_ds, val_idx.tolist()),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)


## 4. Mr.X R-GNN model

In [ ]:
class DenseRGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim, n_relations, dropout=0.1):
        super().__init__()
        self.W0 = nn.Linear(in_dim, out_dim, bias=True)
        self.Wr = nn.Parameter(torch.empty(n_relations, in_dim, out_dim))
        nn.init.xavier_uniform_(self.Wr)
        self.dropout = nn.Dropout(dropout)
        self.residual = in_dim == out_dim

    def forward(self, h, adj):
        self_msg = self.W0(h)
        ah = torch.einsum('rmn,bnk->rbmk', adj, h)
        ahw = torch.einsum('rbmk,rkj->rbmj', ah, self.Wr)
        out = F.relu(self_msg + ahw.sum(dim=0))
        out = self.dropout(out)
        if self.residual:
            out = out + h
        return out


class MrXRGNN(nn.Module):
    def __init__(
        self,
        node_dyn_dim,
        node_static_dim,
        global_dim,
        hidden=96,
        n_layers=3,
        n_relations=4,
        edge_emb_dim=12,
        dropout=0.10,
    ):
        super().__init__()
        self.hidden = hidden
        self.n_relations = n_relations
        self.input_proj = nn.Linear(node_dyn_dim + node_static_dim, hidden)
        self.layers = nn.ModuleList([
            DenseRGCNLayer(hidden, hidden, n_relations, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.edge_type_emb = nn.Embedding(n_relations, edge_emb_dim)
        policy_in = 2 * hidden + edge_emb_dim + global_dim
        self.policy_mlp = nn.Sequential(
            nn.Linear(policy_in, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
        value_in = 2 * hidden + global_dim
        self.value_mlp = nn.Sequential(
            nn.Linear(value_in, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def encode(self, node_dyn, node_static_b, adj):
        h = self.input_proj(torch.cat([node_dyn, node_static_b], dim=-1))
        for layer in self.layers:
            h = layer(h, adj)
        return h

    def forward(self, batch, adj, node_static):
        node_dyn = batch['node_dyn']
        glob = batch['glob']
        mrx_pos_idx = batch['mrx_pos_idx']
        batch_size = node_dyn.shape[0]
        n_nodes = node_dyn.shape[1]

        node_static_b = node_static.unsqueeze(0).expand(batch_size, -1, -1)
        h = self.encode(node_dyn, node_static_b, adj)

        b_idx = torch.arange(batch_size, device=h.device)
        h_origin = h[b_idx, mrx_pos_idx]
        h_origin = h_origin[:, None, None, :].expand(batch_size, n_nodes, self.n_relations, -1)
        h_dest = h[:, :, None, :].expand(batch_size, n_nodes, self.n_relations, -1)
        edge_ids = torch.arange(self.n_relations, device=h.device)
        edge_emb = self.edge_type_emb(edge_ids)[None, None, :, :].expand(batch_size, n_nodes, -1, -1)
        glob_exp = glob[:, None, None, :].expand(batch_size, n_nodes, self.n_relations, -1)

        feat = torch.cat([h_origin, h_dest, edge_emb, glob_exp], dim=-1)
        logits = self.policy_mlp(feat).squeeze(-1).reshape(batch_size, n_nodes * self.n_relations)

        h_mean = h.mean(dim=1)
        h_max = h.max(dim=1).values
        value = self.value_mlp(torch.cat([h_mean, h_max, glob], dim=-1)).squeeze(-1)
        return logits, value


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

dense_adj_t = torch.from_numpy(DENSE_ADJ).to(DEVICE)
node_static_t = torch.from_numpy(NODE_STATIC).to(DEVICE)

model = MrXRGNN(
    node_dyn_dim=NODE_DYN_DIM,
    node_static_dim=NODE_STATIC.shape[1],
    global_dim=GLOBAL_DIM,
    hidden=96,
    n_layers=3,
    n_relations=N_REL,
    edge_emb_dim=12,
    dropout=0.10,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model params: {n_params:,}')


## 5. Train

In [ ]:
EPOCHS = 18
LR = 3e-4
WEIGHT_DECAY = 1e-4
VALUE_COEF = 0.25
ENTROPY_COEF = 0.001
GRAD_CLIP = 1.0

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
best_path = OUT_DIR / f'mrx_rgnn_bc_best_{RUN_TAG}.pt'
last_path = OUT_DIR / f'mrx_rgnn_bc_last_{RUN_TAG}.pt'
history_path = OUT_DIR / f'mrx_rgnn_bc_history_{RUN_TAG}.csv'


def move_batch(batch):
    return {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}


def run_epoch(loader, train=True):
    model.train(train)
    totals = defaultdict(float)
    count = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            batch = move_batch(batch)
            logits, value = model(batch, dense_adj_t, node_static_t)
            legal = batch['legal_mask'].bool()
            target = batch['target_action']
            returns = batch['return_to_go']

            masked_logits = logits.masked_fill(~legal, -1e9)
            policy_loss = F.cross_entropy(masked_logits, target)
            value_loss = F.mse_loss(value, returns)
            logp = F.log_softmax(masked_logits, dim=-1)
            probs = logp.exp()
            entropy = -(probs * logp).sum(dim=-1).mean()
            loss = policy_loss + VALUE_COEF * value_loss - ENTROPY_COEF * entropy

            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()

            bs = target.shape[0]
            pred = masked_logits.argmax(dim=-1)
            top3 = masked_logits.topk(3, dim=-1).indices
            count += bs
            totals['loss'] += float(loss.item()) * bs
            totals['policy_loss'] += float(policy_loss.item()) * bs
            totals['value_loss'] += float(value_loss.item()) * bs
            totals['entropy'] += float(entropy.item()) * bs
            totals['acc'] += float((pred == target).float().sum().item())
            totals['top3_acc'] += float((top3 == target[:, None]).any(dim=1).float().sum().item())
            totals['ticket_acc'] += float(((pred % N_REL) == (target % N_REL)).float().sum().item())
            totals['target_logprob'] += float(logp.gather(1, target[:, None]).mean().item()) * bs

    return {k: v / max(count, 1) for k, v in totals.items()}


from collections import defaultdict

history = []
best_val = float('inf')

checkpoint_config = {
    'node_dyn_dim': NODE_DYN_DIM,
    'node_static_dim': int(NODE_STATIC.shape[1]),
    'global_dim': GLOBAL_DIM,
    'hidden': 96,
    'n_layers': 3,
    'n_relations': N_REL,
    'edge_emb_dim': 12,
    'dropout': 0.10,
    'action_dim': ACTION_DIM,
    'relations': RELATIONS,
    'return_mean': return_mean,
    'return_std': return_std,
    'node_dyn_features': [
        'is_mrx',
        'is_detective_0', 'is_detective_1', 'is_detective_2', 'is_detective_3', 'is_detective_4',
        'belief',
        'is_revealed_mrx_node',
        'min_dist_to_detectives',
        'dist_to_detective_0', 'dist_to_detective_1', 'dist_to_detective_2', 'dist_to_detective_3', 'dist_to_detective_4',
    ],
    'global_features': [
        'turn_norm', 'turns_to_next_reveal_norm', 'is_current_reveal_turn', 'is_next_reveal_turn',
        'mrx_taxi', 'mrx_bus', 'mrx_underground', 'mrx_water',
        'det0_taxi', 'det0_bus', 'det0_underground',
        'det1_taxi', 'det1_bus', 'det1_underground',
        'det2_taxi', 'det2_bus', 'det2_underground',
        'det3_taxi', 'det3_bus', 'det3_underground',
        'det4_taxi', 'det4_bus', 'det4_underground',
        'last_ticket_none', 'last_ticket_taxi', 'last_ticket_bus', 'last_ticket_underground', 'last_ticket_water',
    ],
}

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(train_loader, train=True)
    val_metrics = run_epoch(val_loader, train=False)
    scheduler.step()

    row = {'epoch': epoch, 'lr': scheduler.get_last_lr()[0]}
    row.update({f'train_{k}': v for k, v in train_metrics.items()})
    row.update({f'val_{k}': v for k, v in val_metrics.items()})
    history.append(row)

    print(
        f"epoch {epoch:02d} | "
        f"train loss={train_metrics['loss']:.4f} acc={train_metrics['acc']*100:.1f}% top3={train_metrics['top3_acc']*100:.1f}% | "
        f"val loss={val_metrics['loss']:.4f} acc={val_metrics['acc']*100:.1f}% top3={val_metrics['top3_acc']*100:.1f}% "
        f"ticket={val_metrics['ticket_acc']*100:.1f}%"
    )

    if val_metrics['loss'] < best_val:
        best_val = val_metrics['loss']
        torch.save(
            {
                'model_state_dict': model.state_dict(),
                'config': checkpoint_config,
                'epoch': epoch,
                'val_loss': float(val_metrics['loss']),
                'val_acc': float(val_metrics['acc']),
                'val_top3_acc': float(val_metrics['top3_acc']),
                'train_loss': float(train_metrics['loss']),
                'dataset_manifest': manifest,
                'data_dir': str(DATA_DIR),
                'run_tag': RUN_TAG,
            },
            best_path,
        )
        print('  saved best:', best_path)

torch.save(
    {
        'model_state_dict': model.state_dict(),
        'config': checkpoint_config,
        'epoch': EPOCHS,
        'history': history,
        'dataset_manifest': manifest,
        'data_dir': str(DATA_DIR),
        'run_tag': RUN_TAG,
    },
    last_path,
)

hist_df = pd.DataFrame(history)
hist_df.to_csv(history_path, index=False)
print('Best checkpoint:', best_path)
print('Last checkpoint:', last_path)
print('History:', history_path)


## 6. Curves and final checks

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='train')
axes[0].plot(hist_df['epoch'], hist_df['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(hist_df['epoch'], hist_df['train_acc'] * 100, label='train acc')
axes[1].plot(hist_df['epoch'], hist_df['val_acc'] * 100, label='val acc')
axes[1].plot(hist_df['epoch'], hist_df['val_top3_acc'] * 100, label='val top3')
axes[1].set_title('Policy accuracy')
axes[1].set_ylabel('%')
axes[1].legend()

axes[2].plot(hist_df['epoch'], hist_df['val_ticket_acc'] * 100, label='ticket acc')
axes[2].plot(hist_df['epoch'], hist_df['val_entropy'], label='entropy')
axes[2].set_title('Ticket / entropy')
axes[2].legend()

plt.tight_layout()
curves_path = OUT_DIR / f'mrx_rgnn_bc_curves_{RUN_TAG}.png'
plt.savefig(curves_path, dpi=160)
plt.show()

print('Saved curves:', curves_path)
print('Checkpoint files:')
for p in sorted(OUT_DIR.glob(f'*{RUN_TAG}*')):
    print(' ', p, p.stat().st_size)

best_row = hist_df.loc[hist_df['val_loss'].idxmin()].to_dict()
print('Best epoch summary:')
print(json.dumps({k: float(v) if isinstance(v, (int, float, np.integer, np.floating)) else v for k, v in best_row.items()}, indent=2))
